In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import requests
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
load_dotenv()

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


True

In [3]:
huggingfacehub_api_token = os.getenv('HUGGINGFACEHUB_ACCESS_TOKEN')
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    huggingfacehub_api_token=huggingfacehub_api_token,
)
model = ChatHuggingFace(llm=llm)
search_tool = DuckDuckGoSearchRun()

In [4]:
from langchain_core.tools import tool
weather_api_key = os.getenv('WEATHER_API_KEY')
@tool
def fetch_weather_data(city: str) -> dict:
    """The function takes in input, a city and finds out the weather for that city"""
    response = requests.get(f'https://api.weatherstack.com/current?access_key={weather_api_key}&query={city}')
    response = response.json()
    
    weather_data = {
        "city": response["location"]["name"],
        "temperature": response["current"]["temperature"],
        "description": response["current"]["weather_descriptions"][0],
        "humidity": response["current"]["humidity"],
        "wind_speed": response["current"]["wind_speed"],
        "uv_index": response["current"]["uv_index"]
    }
    return weather_data

In [5]:
import json
@tool
def multiply(input: str) -> float:
    """
    Multiplies two numbers.
    Input format:
    {"val1": number, "val2": number}
    """
    data = json.loads(input)
    return float(data["val1"]) * float(data["val2"])

In [6]:
# Agent imports
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub

# Step 2: Pull the ReAct (Reasoning + Action) prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")  # pulls the standard ReAct agent prompt
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [7]:
agent = create_react_agent(
    llm = model,
    tools = [search_tool, fetch_weather_data, multiply],
    prompt = prompt
) # agent who plans task and all

In [ ]:
# Now we need to wrap it with AgentExecutor (It is the one that executes what the agent says)
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool, fetch_weather_data, multiply],
    # max_iterations=3, # limits the agent to at most 3 thought–action loops before stopping
    # early_stopping_method="generate", # if the agent hits the iteration limit, force the LLM to generate a final answer from the gathered observations
    verbose = True
)

In [9]:
response = agent_executor.invoke({
    "input": "Find the weather conditions of the financial capital of india and then multiply the temperature with 25. Also give me the population of that city."
})['output']
print(response)



> Entering new AgentExecutor chain...
I need to find the financial capital of India first, then get its weather data, multiply the temperature by 25, and also find the population of the city.
Action: duckduckgo_search
Action Input: Financial capital of IndiaIndiasFinancialCapital: Your Gateway to Seamless Services ... In the bustling metropolisofMumbai, where tradition seamlessly blends with ... CapitalofIndia: Are you looking for information about theCapitalofIndia? In this page we have given the 29 statesofIndiaalong withcapital... CapitalIndiaFinance Ltd, a non-bankingfinancialcompany, has made its debut on the National Stock ExchangeofIndia(NSE) in a strategic move to ... Mumbai isn ’ t just thefinancialcapitalofIndia; it ’ s a living, breathing testament to what can happen when human potential meets ... CapitalIndiaFinance Limited , a leading non-bankingfinancialcompany , has marked a significant milestone by listing its equity shares on the ...From the search results, it is cle